# Linkage Probabilístico SIM × Sinasc

**Referência metodológica:** Maia LTS et al., *Rev Saude Publica*. 2017;51:112  
**Algoritmo:** Fellegi-Sunter via biblioteca `recordlinkage`

---

### Etapas do pipeline
1. Carga dos dados
2. Pré-processamento
3. Blocagem
4. Comparação campo a campo
5. Classificação dos pares
6. Desambiguação (one-to-one)
7. Merge final
8. Plot da distribuição dos escores
9. Exportação
10. Análise de qualidade

## Instalação das dependências

In [4]:
# Execute esta célula apenas na primeira vez
%pip install recordlinkage pandas matplotlib

   ---------------------------------------- 0.0/926.9 kB ? eta -:--:--
   ---------------------------------------- 926.9/926.9 kB 8.5 MB/s  0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ----------------------- ---------------- 4.7/8.1 MB 23.8 MB/s eta 0:00:01
   ----------------------------------- ---- 7.1/8.1 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 17.3 MB/s  0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ------ --------------------------------- 6.3/36.6 MB 32.1 MB/s eta 0:00:01
   ------------ --------------------------- 11.8/36.6 MB 28.3 MB/s eta 0:00:01
   -------------------- ------------------- 18.9/36.6 MB 29.8 MB/s eta 0:00:01
   --------------------------- ------------ 25.4/36.6 MB 30.4 MB/s eta 0:00:01
   --------------------------------- ------ 30.4/36.6 MB 28.8 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 29.3 MB/s eta 0:00:01
   ---------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## ⚙️ Configuração
**Ajuste os caminhos e o limiar antes de rodar o notebook.**

In [43]:
# Caminhos para os arquivos de dados (CSV ou DBF convertido)
CAMINHO_SIM    = "DO23OPEN.csv"    
CAMINHO_SINASC = "SINASC_2023.csv" 

# Limiar de escore para classificar como match
# Ajuste após visualizar o histograma na etapa 8
LIMIAR_MATCH = 6.0

# Campos de blocagem (reduzem o espaço de comparação)
CAMPO_BLOCO_1 = "CODMUNRES"  # município de residência
CAMPO_BLOCO_2 = "mes_nasc"   # mês de nascimento (derivado de DTNASC)

## 0. Imports

In [44]:
import pandas as pd
import recordlinkage
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


## 1. Carga dos dados

In [45]:
sim    = pd.read_csv(CAMINHO_SIM,    dtype=str, encoding="latin-1", sep=";")
sinasc = pd.read_csv(CAMINHO_SINASC, dtype=str, encoding="latin-1", sep=";")

# Normaliza nomes de coluna
sim.columns    = sim.columns.str.upper().str.strip()
sinasc.columns = sinasc.columns.str.upper().str.strip()

print(f"SIM:    {len(sim):,} registros | {sim.shape[1]} colunas")
print(f"Sinasc: {len(sinasc):,} registros | {sinasc.shape[1]} colunas")

SIM:    1,465,610 registros | 86 colunas
Sinasc: 2,537,576 registros | 62 colunas


In [46]:
# Prévia das colunas disponíveis
print("Colunas SIM:   ", list(sim.columns[:15]), "...")
print("Colunas Sinasc:", list(sinasc.columns[:15]), "...")

Colunas SIM:    ['CONTADOR', 'ORIGEM', 'TIPOBITO', 'DTOBITO', 'HORAOBITO', 'NATURAL', 'CODMUNNATU', 'DTNASC', 'IDADE', 'SEXO', 'RACACOR', 'ESTCIV', 'ESC', 'ESC2010', 'SERIESCFAL'] ...
Colunas Sinasc: ['CONTADOR', 'ORIGEM', 'CODESTAB', 'CODMUNNASC', 'LOCNASC', 'IDADEMAE', 'ESTCIVMAE', 'ESCMAE', 'CODOCUPMAE', 'QTDFILVIVO', 'QTDFILMORT', 'CODMUNRES', 'GESTACAO', 'GRAVIDEZ', 'PARTO'] ...


## 2. Pré-processamento

In [47]:
def preprocessar(df, origem):
    df = df.copy()

    # Data de nascimento → datetime; extrai mês e ano para blocagem
    df['DTNASC'] = df['DTNASC'].astype(str).str.zfill(8)
    df["DTNASC"]   = pd.to_datetime(df["DTNASC"], format="%d%m%Y", errors="coerce")
    df["mes_nasc"] = df["DTNASC"].dt.month
    df["ano_nasc"] = df["DTNASC"].dt.year

    # Campos numéricos: remove espaços e padroniza com zfill
    for col in ["IDADEMAE", "PESO", "GESTACAO", "SEMAGESTAC"]:
        if col in df.columns:
            df[col] = df[col].str.strip().str.zfill(4)

    # Campos categóricos: uppercase sem espaços
    for col in ["SEXO", "PARTO", "GRAVIDEZ"]:
        if col in df.columns:
            df[col] = df[col].str.strip().str.upper()

    # CODMUNRES: garante 7 dígitos
    if "CODMUNRES" in df.columns:
        df["CODMUNRES"] = df["CODMUNRES"].str.strip().str.zfill(7)

    print(f"[{origem}] {len(df):,} registros pré-processados.")
    return df

sim    = preprocessar(sim,    "SIM")
sinasc = preprocessar(sinasc, "SINASC")

[SIM] 1,465,610 registros pré-processados.
[SINASC] 2,537,576 registros pré-processados.


In [48]:
sim["faixa"] = sim["IDADE"].str[0]
sim["quant"] = sim["IDADE"].str[1:].astype(int)

sim = sim[
    (sim["faixa"].isin(["1", "2"])) |               # minutos e horas
    ((sim["faixa"] == "3") & (sim["quant"] <= 28))  # dias até 28
]

In [49]:
# Checar campos que serão usados no linkage
campos_linkage = ["DTNASC", "PESO", "SEXO", "IDADEMAE", "SEMAGESTAC", "GESTACAO", "PARTO", "GRAVIDEZ", "CODMUNRES", "RACACOR"]

print("Campo         | SIM  | Sinasc | % nulos SIM | % nulos Sinasc")
print("-" * 65)
for campo in campos_linkage:
    em_sim    = campo in sim.columns
    em_sinasc = campo in sinasc.columns
    nulo_sim    = f"{sim[campo].isna().mean()*100:.1f}%"    if em_sim    else "N/A"
    nulo_sinasc = f"{sinasc[campo].isna().mean()*100:.1f}%" if em_sinasc else "N/A"
    print(f"{campo:<14}| {'✓' if em_sim else '✗'}    | {'✓' if em_sinasc else '✗'}      | {nulo_sim:<11} | {nulo_sinasc}")

Campo         | SIM  | Sinasc | % nulos SIM | % nulos Sinasc
-----------------------------------------------------------------
DTNASC        | ✓    | ✓      | 0.5%        | 0.0%
PESO          | ✓    | ✓      | 10.8%       | 0.0%
SEXO          | ✓    | ✓      | 0.0%        | 0.0%
IDADEMAE      | ✓    | ✓      | 9.2%        | 0.0%
SEMAGESTAC    | ✓    | ✓      | 10.8%       | 0.8%
GESTACAO      | ✓    | ✓      | 10.8%       | 0.8%
PARTO         | ✓    | ✓      | 7.3%        | 0.0%
GRAVIDEZ      | ✓    | ✓      | 6.9%        | 0.1%
CODMUNRES     | ✓    | ✓      | 0.0%        | 0.0%
RACACOR       | ✓    | ✓      | 5.9%        | 1.8%


## 3. Blocagem

A blocagem restringe as comparações apenas a pares que compartilham **município de residência** e **mês de nascimento**, reduzindo o espaço de busca de O(n²) para algo gerenciável sem perder recall relevante.

In [50]:
indexer = recordlinkage.Index()
indexer.block([CAMPO_BLOCO_1, CAMPO_BLOCO_2])

candidatos = indexer.index(sim, sinasc)

total_possivel = len(sim) * len(sinasc)
reducao = (1 - len(candidatos) / total_possivel) * 100

print(f"Pares candidatos após blocagem: {len(candidatos):,}")
print(f"Total de pares possíveis:       {total_possivel:,}")
print(f"Redução pelo bloqueio:          {reducao:.1f}%")

Pares candidatos após blocagem: 29,040,431
Total de pares possíveis:       74,548,907,728
Redução pelo bloqueio:          100.0%


## 4. Comparação campo a campo

Cada campo recebe pesos positivos (concordância) e negativos (discordância) de acordo com seu poder discriminatório:

| Campo | Concordância | Discordância | Justificativa |
|---|---|---|---|
| DTNASC | +3 | −2 | Alta especificidade dentro do bloco |
| PESO | +3 | −1 | Alta variabilidade contínua |
| IDADEMAE | +2 | −1 | Boa discriminação |
| SEXO | +1 | −1 | Binário, menor discriminação |
| GESTACAO/SEMAGESTAC | +1 | 0 | Frequentemente incompleto |
| PARTO / GRAVIDEZ | +1 | 0 | Baixa variabilidade |

In [51]:
compare = recordlinkage.Compare()

compare.exact("DTNASC",   "DTNASC",   label="dtnasc",   agree_value=3,  disagree_value=-2)
compare.exact("PESO",     "PESO",     label="peso",     agree_value=3,  disagree_value=-1, missing_value=0)
compare.exact("SEXO",     "SEXO",     label="sexo",     agree_value=1,  disagree_value=-1)
compare.exact("IDADEMAE", "IDADEMAE", label="idademae", agree_value=2,  disagree_value=-1)

compare.exact("RACACOR", "RACACOR", label="racacor", agree_value=1, disagree_value=-1, missing_value=0)

# SEMAGESTAC: concordância parcial
compare.exact("SEMAGESTAC", "SEMAGESTAC", label="semanas_gestacao", agree_value=2, disagree_value=0, missing_value=0)

if "PARTO"   in sim.columns and "PARTO"   in sinasc.columns:
    compare.exact("PARTO",   "PARTO",   label="parto",   agree_value=1, disagree_value=0, missing_value=0)
if "GRAVIDEZ" in sim.columns and "GRAVIDEZ" in sinasc.columns:
    compare.exact("GRAVIDEZ", "GRAVIDEZ", label="gravidez", agree_value=1, disagree_value=0, missing_value=0)

features = compare.compute(candidatos, sim, sinasc)
features["escore"] = features.sum(axis=1)

print("Comparação concluída.")
print(f"\nDistribuição dos escores (maiores valores):")
features["escore"].value_counts().sort_index(ascending=False).head(10)

Comparação concluída.

Distribuição dos escores (maiores valores):


escore
14      6923
13      1019
12      6905
11      1492
10      3298
9       1797
8       5493
7      10321
6      21967
5     108017
Name: count, dtype: int64

## 5. Classificação dos pares

In [52]:
matches = features[features["escore"] >= LIMIAR_MATCH].copy()

print(f"Pares classificados como match (escore ≥ {LIMIAR_MATCH}): {len(matches):,}")
print(f"Pares descartados como non-match:                         {len(features) - len(matches):,}")

Pares classificados como match (escore ≥ 6.0): 59,215
Pares descartados como non-match:                         28,981,216


## 6. Desambiguação — garante one-to-one

Para cada registro do SIM, mantém apenas o par de **maior escore** no Sinasc, e vice-versa. Isso elimina a ambiguidade antes do merge final, tornando o `validate='one_to_one'` uma simples confirmação de integridade.

In [53]:
matches = matches.reset_index()
matches.columns = ["idx_sim", "idx_sinasc"] + list(matches.columns[2:])

antes = len(matches)

# Melhor match por registro SIM
matches = matches.loc[matches.groupby("idx_sim")["escore"].idxmax()]
# Melhor match por registro Sinasc
matches = matches.loc[matches.groupby("idx_sinasc")["escore"].idxmax()]

print(f"Pares antes da desambiguação:  {antes:,}")
print(f"Pares após desambiguação:      {len(matches):,}")
print(f"Pares removidos por ambiguidade: {antes - len(matches):,}")

Pares antes da desambiguação:  59,215
Pares após desambiguação:      22,795
Pares removidos por ambiguidade: 36,420


## 7. Merge final

O `validate='one_to_one'` aqui é uma **assertion de segurança**: se a etapa 6 funcionou corretamente, ele passa silenciosamente. Caso contrário, levanta `MergeError` imediatamente em vez de produzir um dataset corrompido.

In [54]:
sim_matched    = sim.loc[matches["idx_sim"]].reset_index(drop=True)
sinasc_matched = sinasc.loc[matches["idx_sinasc"]].reset_index(drop=True)

sim_out    = sim_matched.add_suffix("_SIM").copy()
sinasc_out = sinasc_matched.add_suffix("_SINASC").copy()
sim_out["_key"]    = range(len(sim_out))
sinasc_out["_key"] = range(len(sinasc_out))

resultado = pd.merge(
    sim_out,
    sinasc_out,
    on="_key",
    validate="one_to_one",  # assertion: confirma one-to-one após etapa 6
).drop(columns="_key")
resultado["escore"] = matches["escore"].values

print(f"✓ Merge one-to-one validado com sucesso.")
print(f"\nDataset final: {len(resultado):,} pares linkados")
print(f"Taxa de linkagem SIM:    {len(resultado)/len(sim)*100:.1f}%")
print(f"Taxa de linkagem Sinasc: {len(resultado)/len(sinasc)*100:.1f}%")

✓ Merge one-to-one validado com sucesso.

Dataset final: 22,795 pares linkados
Taxa de linkagem SIM:    77.6%
Taxa de linkagem Sinasc: 0.9%


## 10. Análise de qualidade

In [55]:
# Contribuição de cada campo para os matches
campos_compare = [c for c in features.columns if c != "escore"]
print("Concordância por campo (% dos pares matched):")
matches_features = features.loc[
    features.index.isin(list(zip(matches["idx_sim"], matches["idx_sinasc"])))
]
for campo in campos_compare:
    if campo in matches_features.columns:
        concordancia = (matches_features[campo] > 0).mean() * 100
        print(f"  {campo:<12} {concordancia:.1f}%")

Concordância por campo (% dos pares matched):
  dtnasc       98.9%
  peso         85.8%
  sexo         98.6%
  idademae     87.3%
  racacor      65.5%
  semanas_gestacao 62.9%
  parto        95.7%
  gravidez     97.8%


## 11. Criação do CSV final

In [56]:
# Índices do sinasc que deram match
idx_sinasc_matched = matches["idx_sinasc"].values

sinasc['obitos_neo'] = 0
sinasc.loc[idx_sinasc_matched, 'obitos_neo'] = 1

print(sinasc['obitos_neo'].value_counts())

obitos_neo
0    2514781
1      22795
Name: count, dtype: int64


In [57]:
# Seleciona todas as linhas com óbito = 1
obitos_1 = sinasc[sinasc['obitos_neo'] == 1]

# Seleciona a mesma quantidade de linhas com óbito = 0 (amostragem aleatória)
n_tam = len(obitos_1)
obitos_0 = sinasc[sinasc['obitos_neo'] == 0].sample(n=n_tam, random_state=42)

# Junta os dois DataFrames
df_final = pd.concat([obitos_1, obitos_0])

In [58]:
df_final['obitos_neo'].value_counts()

obitos_neo
1    22795
0    22795
Name: count, dtype: int64

In [59]:
df_final.to_csv("sinasc_obito_2023.csv", index=False, encoding="utf-8-sig", sep=";")

In [61]:
df_final[df_final['obitos_neo'] == 1]

,CONTADOR,ORIGEM,CODESTAB,CODMUNNASC,LOCNASC,IDADEMAE,ESTCIVMAE,ESCMAE,CODOCUPMAE,QTDFILVIVO,...,ESCMAEAGR1,STDNEPIDEM,STDNNOVA,CODPAISRES,TPROBSON,PARIDADE,KOTELCHUCK,mes_nasc,ano_nasc,obitos_neo
362,363,1,2494299,110002,1,0024,2,4,999992,00,...,05,0,1,1,10,1,3,3,2023,1
440,441,1,2494299,110002,1,0033,4,4,512105,02,...,06,0,1,1,05,1,5,3,2023,1
676,677,1,2515601,110002,1,0038,4,5,351715,01,...,08,0,1,1,10,1,5,4,2023,1
685,686,1,2515504,110002,1,0033,2,4,999992,01,...,06,0,1,1,07,1,5,4,2023,1
851,852,1,2494299,110002,1,0031,5,4,999992,05,...,06,0,1,1,03,1,5,6,2023,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2536584,2536585,1,2815966,530010,1,0040,1,5,516115,01,...,08,1,1,1,10,1,3,12,2023,1
2536918,2536919,1,7539797,530010,1,0028,2,4,521110,00,...,12,1,1,1,06,0,3,12,2023,1
2537001,2537002,1,7539797,530010,1,0037,2,5,999991,01,...,08,1,1,1,05,1,5,12,2023,1
2537193,2537194,1,2672197,530010,1,0029,1,5,516120,01,...,07,1,1,1,10,1,2,10,2023,1
